# Exploratory analysis — dengue case counts (SINAN)

Phase 1 starts here: understand the raw extract before aggregating to UF × epidemiological week.

**This notebook (for now):** project setup, load the SINAN extract, and describe its grain and fields.

**Not yet:** aggregation, heatmaps, or baseline modelling.

## Setup

Resolve paths from `config.yml` so the notebook stays portable relative to the repo root.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

REPO_ROOT = Path("..").resolve()
CONFIG_PATH = REPO_ROOT / "config.yml"

with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

SINAN_PATH = REPO_ROOT / config["data"]["sinan"]["path"]

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"repo:  {REPO_ROOT}")
print(f"sinan: {SINAN_PATH}")
assert SINAN_PATH.exists(), f"Missing extract: {SINAN_PATH}"

## Load

One row is a count of dengue notifications sharing the same state, epidemiological-week timestamps, and final classification — not an individual case record.

In [ ]:
cases = pd.read_parquet(SINAN_PATH)

# Parquet stores classification as nullable int8; pandas widens it to float when nulls appear.
cases["final_classification"] = cases["final_classification"].astype("Int8")

cases.shape

## What this table is

SINAN dengue notifications, already rolled up to weekly counts by federative unit (UF).

| Field | Meaning |
|---|---|
| `ew_symptom_onset` | Monday of the epidemiological week of symptom onset (`DT_SIN_PRI`) — the date field used for forecasting |
| `ew_notification` | Monday of the week the case was notified |
| `ew_recorded` | Monday of the week the record entered the database (can be missing or corrupted) |
| `state_abbrev` | UF abbreviation (27 units) |
| `final_classification` | SINAN `CLASSI_FIN` code |
| `case_count` | Number of notifications in that cell |

**Forecasting target (from project config):** probable cases by UF and symptom-onset week — notified minus discarded (`CLASSI_FIN ≠ 5`), window from 2010-EW01 onward, with the final 12 weeks of the extract dropped for right-censoring.

### Classification codes (`CLASSI_FIN`)

| Code | Label |
|---:|---|
| 1 | Dengue (classic / older coding) |
| 2 | Dengue with complications |
| 3 | Dengue haemorrhagic fever |
| 4 | Dengue shock syndrome |
| 5 | Discarded |
| 8 | Inconclusive |
| 10 | Dengue |
| 11 | Dengue with warning signs |
| 12 | Severe dengue |
| — | Missing / unused codes appear rarely |

## Portrait of the extract

A few load-bearing facts — size, coverage, and how cases split by classification — before any plotting or aggregation.

In [ ]:
onset = pd.to_datetime(cases["ew_symptom_onset"])
notification = pd.to_datetime(cases["ew_notification"])
recorded = pd.to_datetime(cases["ew_recorded"], errors="coerce")

total_cases = int(cases["case_count"].sum())
n_ufs = cases["state_abbrev"].nunique()

portrait = pd.Series(
    {
        "rows": f"{len(cases):,}",
        "total_notifications": f"{total_cases:,}",
        "federative_units": n_ufs,
        "onset_weeks": f"{onset.min().date()} → {onset.max().date()}",
        "notification_weeks": f"{notification.min().date()} → {notification.max().date()}",
        "missing_recorded_week": f"{cases['ew_recorded'].isna().mean():.1%}",
        "missing_classification": f"{cases['final_classification'].isna().mean():.1%}",
    },
    name="value",
).to_frame()

portrait

In [ ]:
CLASS_LABELS = {
    1: "Dengue (classic)",
    2: "Dengue with complications",
    3: "Dengue haemorrhagic fever",
    4: "Dengue shock syndrome",
    5: "Discarded",
    8: "Inconclusive",
    10: "Dengue",
    11: "Dengue with warning signs",
    12: "Severe dengue",
}

by_class = (
    cases.groupby("final_classification", dropna=False, observed=True)["case_count"]
    .sum()
    .rename("notifications")
    .to_frame()
)
by_class["share"] = by_class["notifications"] / by_class["notifications"].sum()
by_class["label"] = by_class.index.map(
    lambda code: CLASS_LABELS.get(int(code), "Other / missing") if pd.notna(code) else "Missing"
)
by_class = by_class[["label", "notifications", "share"]].sort_values("notifications", ascending=False)
by_class["notifications"] = by_class["notifications"].map("{:,.0f}".format)
by_class["share"] = by_class["share"].map("{:.1%}".format)
by_class

In [ ]:
discarded = cases["final_classification"].eq(5)
probable_mask = ~discarded  # notified − discarded (roadmap case definition)

probable_cases = int(cases.loc[probable_mask, "case_count"].sum())
discarded_cases = int(cases.loc[discarded, "case_count"].sum())

pd.Series(
    {
        "probable (keep)": f"{probable_cases:,} ({probable_cases / total_cases:.1%})",
        "discarded (drop)": f"{discarded_cases:,} ({discarded_cases / total_cases:.1%})",
    },
    name="notifications",
).to_frame()

In [ ]:
by_uf = (
    cases.loc[probable_mask]
    .groupby("state_abbrev", observed=True)["case_count"]
    .sum()
    .sort_values(ascending=False)
    .rename("probable_notifications")
)

pd.DataFrame(
    {
        "probable_notifications": by_uf.map("{:,.0f}".format),
        "share_of_national": (by_uf / by_uf.sum()).map("{:.1%}".format),
    }
)

### Reading notes

- **Grain is already weekly × UF × classification.** Aggregation to the forecasting panel mostly means filtering classifications, summing `case_count`, and aligning to a complete EW calendar.
- **Symptom onset is the temporal axis** for models; notification and recording weeks matter later for delay / censoring diagnostics, not for the primary series.
- **`ew_recorded` is messy** (~18% missing; a handful of impossible dates). Do not use it as the time index.
- **Burden is heavily skewed across UFs** (São Paulo and Minas Gerais dominate national totals). That is why headline metrics will be computed on `log1p` scale.

Next: build `data/processed/dengue_uf_ew.parquet` under the aggregation rules in `config.yml`.